# Residential Units ETL — QA & Summary Analysis

Reads the output feature class and QA tables from `C:\GIS\ParcelHistory.gdb`  
to validate the ETL results and summarize unit counts by geography.

**Run order:** run the ETL (`main.py`) before opening this notebook.

In [ ]:
import sys
sys.path.insert(0, r'C:\Users\mbindl\Documents\GitHub\Reporting\residential_units_etl')

import arcpy
import pandas as pd
import numpy as np

from config import (
    OUTPUT_FC, CSV_YEARS, FC_APN, FC_YEAR, FC_UNITS, FC_COUNTY,
    GDB,
    QA_UNITS_BY_YEAR, QA_LOST_APNS, QA_APN_CROSSWALK,
    QA_SPATIAL_COMPLETENESS,
    SPATIAL_FIELDS,
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.1f}'.format)
print('Setup complete')

---
## 1. Load Output Feature Class

In [ ]:
GEO_FIELDS = [
    'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER',
    'TAZ', 'PLAN_ID', 'PLAN_NAME',
    'ZONING_ID', 'ZONING_DESCRIPTION', 'REGIONAL_LANDUSE',
    'WITHIN_TRPA_BNDY', 'WITHIN_BONUSUNIT_BNDY',
    'PARCEL_ACRES',
]

READ_FIELDS = [FC_APN, FC_YEAR, FC_UNITS, FC_COUNTY] + GEO_FIELDS

# Only read fields that actually exist
existing = {f.name for f in arcpy.ListFields(OUTPUT_FC)}
missing  = [f for f in READ_FIELDS if f not in existing]
if missing:
    print(f'WARNING — fields not in FC (skipped): {missing}')
read = [f for f in READ_FIELDS if f in existing]

yr_list = ', '.join(str(y) for y in CSV_YEARS)
rows = []
with arcpy.da.SearchCursor(OUTPUT_FC, read, f"{FC_YEAR} IN ({yr_list})") as cur:
    for row in cur:
        rows.append(dict(zip(read, row)))

df = pd.DataFrame(rows).rename(columns={
    FC_APN: 'APN', FC_YEAR: 'Year', FC_UNITS: 'Units', FC_COUNTY: 'County'
})
df['Units'] = df['Units'].fillna(0).astype(int)

print(f'Rows loaded : {len(df):,}')
print(f'Years       : {sorted(df["Year"].unique())}')
print(f'Total Units : {df["Units"].sum():,}')
df.head(3)

In [ ]:
import arcpy, pandas as pd
rows = [r for r in arcpy.da.SearchCursor(
    r"C:\GIS\ParcelHistory.gdb\QA_Lost_APNs", ["APN","Issue_Category","Total_Units_CSV"])]
df = pd.DataFrame(rows, columns=["APN","Issue_Category","Total_Units"])
print(df[df.Issue_Category == "PARCEL_NEW"])

---
## 2. Units by Year — CSV vs FC

In [ ]:
# Load QA table (written by ETL Step 6)
qa_yr_rows = []
with arcpy.da.SearchCursor(QA_UNITS_BY_YEAR,
        ['Year','CSV_Total','FC_Total','Diff','Status']) as cur:
    for r in cur:
        qa_yr_rows.append(dict(zip(['Year','CSV_Total','FC_Total','Diff','Status'], r)))

df_yr = pd.DataFrame(qa_yr_rows).sort_values('Year').reset_index(drop=True)
df_yr['Match'] = df_yr['Diff'].apply(lambda d: '✓' if d == 0 else f'{d:+,}')

print(df_yr[['Year','CSV_Total','FC_Total','Diff','Status']].to_string(index=False))

---
## 3. Units by County & Year

In [ ]:
if 'County' in df.columns:
    piv = (df.groupby(['County','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
    piv['TOTAL'] = piv.sum(axis=1)
    print(piv.sort_values('TOTAL', ascending=False))
else:
    print('County field not loaded')

---
## 4. Units by Town Center & Year

TRPA boundary rows only (excludes out-of-basin parcels).

In [ ]:
if 'TOWN_CENTER' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['TC'] = df_trpa['TOWN_CENTER'].fillna('No Town Center')

    piv_tc = (df_trpa.groupby(['TC','Year'])['Units']
                .sum()
                .unstack('Year')
                .fillna(0)
                .astype(int))
    piv_tc['TOTAL'] = piv_tc.sum(axis=1)
    piv_tc = piv_tc.sort_values('TOTAL', ascending=False)
    print(piv_tc)
else:
    print('TOWN_CENTER or WITHIN_TRPA_BNDY not available')

---
## 5. Units by Location-to-Town-Center & Year

In [ ]:
if 'LOCATION_TO_TOWNCENTER' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['Loc'] = df_trpa['LOCATION_TO_TOWNCENTER'].fillna('Unknown')

    piv_loc = (df_trpa.groupby(['Loc','Year'])['Units']
                 .sum()
                 .unstack('Year')
                 .fillna(0)
                 .astype(int))
    piv_loc['TOTAL'] = piv_loc.sum(axis=1)
    piv_loc = piv_loc.sort_values('TOTAL', ascending=False)
    print(piv_loc)
else:
    print('LOCATION_TO_TOWNCENTER or WITHIN_TRPA_BNDY not available')

---
## 6. Units by Regional Land Use

In [ ]:
if 'REGIONAL_LANDUSE' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['RLU'] = df_trpa['REGIONAL_LANDUSE'].fillna('Unknown')

    piv_rlu = (df_trpa.groupby(['RLU','Year'])['Units']
                 .sum()
                 .unstack('Year')
                 .fillna(0)
                 .astype(int))
    piv_rlu['TOTAL'] = piv_rlu.sum(axis=1)
    piv_rlu = piv_rlu.sort_values('TOTAL', ascending=False)
    print(piv_rlu)
else:
    print('REGIONAL_LANDUSE or WITHIN_TRPA_BNDY not available')

---
## 7. Units by TAZ

In [ ]:
if 'TAZ' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()

    taz_yr = (df_trpa[df_trpa['TAZ'].notna()]
                .groupby(['TAZ','Year'])['Units']
                .sum()
                .unstack('Year')
                .fillna(0)
                .astype(int))
    taz_yr['TOTAL'] = taz_yr.sum(axis=1)
    taz_yr = taz_yr.sort_values('TOTAL', ascending=False)

    print(f'TAZ count: {len(taz_yr)}')
    print(taz_yr.head(20))
else:
    print('TAZ or WITHIN_TRPA_BNDY not available')

---
## 8. Units by Zoning

In [ ]:
df_trpa

In [ ]:
if 'ZONING_ID' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['Zone'] = df_trpa['ZONING_ID'].fillna('Unknown')

    zone_sum = (df_trpa.groupby('Zone')['Units']
                  .sum()
                  .reset_index()
                  .sort_values('Units', ascending=False))
    print(zone_sum.head(20).to_string(index=False))
else:
    print('ZONING_DESCRIPTION or WITHIN_TRPA_BNDY not available')

---
## 9. Spatial Completeness (TRPA boundary rows only)

In [ ]:
if arcpy.Exists(QA_SPATIAL_COMPLETENESS):
    sp_rows = []
    flds = ['Field','Null_Count','Pct_Null','TRPA_Rows','Status']
    with arcpy.da.SearchCursor(QA_SPATIAL_COMPLETENESS, flds) as cur:
        for r in cur:
            sp_rows.append(dict(zip(flds, r)))
    df_sp = pd.DataFrame(sp_rows).sort_values('Pct_Null', ascending=False)
    print(df_sp.to_string(index=False))
else:
    print('QA_Spatial_Completeness table not found — run ETL Step 6 first')

---
## 10. Lost APNs Summary

In [ ]:
if arcpy.Exists(QA_LOST_APNS):
    la_flds = ['APN','COUNTY','Years_Lost','Num_Years_Lost',
               'Total_Units_CSV','Years_In_FC','Issue_Category','Suggested_Action']
    la_rows = []
    with arcpy.da.SearchCursor(QA_LOST_APNS, la_flds) as cur:
        for r in cur:
            la_rows.append(dict(zip(la_flds, r)))
    df_lost = pd.DataFrame(la_rows)

    print(f'Total lost APNs: {len(df_lost):,}')
    print(f'Total lost units: {df_lost["Total_Units_CSV"].sum():,}')
    print()
    cat_sum = (df_lost.groupby('Issue_Category')
                 .agg(APNs=('APN','count'), Units=('Total_Units_CSV','sum'))
                 .sort_values('Units', ascending=False))
    print(cat_sum)
else:
    print('QA_Lost_APNs table not found — run ETL first')

In [ ]:
# Top lost APNs by units
if 'df_lost' in dir() and len(df_lost):
    print('Top 20 lost APNs by units:')
    top = df_lost.nlargest(20, 'Total_Units_CSV')[[
        'APN','COUNTY','Issue_Category','Total_Units_CSV','Years_Lost','Suggested_Action'
    ]]
    pd.set_option('display.max_colwidth', 80)
    print(top.to_string(index=False))

---
## 11. Bonus Unit Area Summary

In [ ]:
if 'WITHIN_BONUSUNIT_BNDY' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1]

    bonus_yr = (df_trpa.groupby(['WITHIN_BONUSUNIT_BNDY','Year'])['Units']
                  .sum()
                  .unstack('Year')
                  .fillna(0)
                  .astype(int))
    bonus_yr.index = bonus_yr.index.map({0: 'Outside Bonus Area', 1: 'Within Bonus Area'})
    bonus_yr['TOTAL'] = bonus_yr.sum(axis=1)
    print(bonus_yr)
else:
    print('WITHIN_BONUSUNIT_BNDY not available')

---
## 12. Year-over-Year Change by Town Center

In [ ]:
if 'TOWN_CENTER' in df.columns and 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1].copy()
    df_trpa['TC'] = df_trpa['TOWN_CENTER'].fillna('No Town Center')

    piv = (df_trpa.groupby(['TC','Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))

    years = sorted(CSV_YEARS)
    chg = pd.DataFrame(index=piv.index)
    for i in range(1, len(years)):
        y0, y1 = years[i-1], years[i]
        if y0 in piv.columns and y1 in piv.columns:
            chg[f'{y0}→{y1}'] = piv[y1] - piv[y0]

    print('Year-over-year unit change by Town Center:')
    print(chg.sort_values(chg.columns[-1], ascending=False))
else:
    print('TOWN_CENTER or WITHIN_TRPA_BNDY not available')

---
## 13. Parcel Count vs Unit Count by Year (density check)

In [ ]:
if 'WITHIN_TRPA_BNDY' in df.columns:
    df_trpa = df[df['WITHIN_TRPA_BNDY'] == 1]

    density = (df_trpa.groupby('Year')
                 .agg(
                     Parcels=('APN','count'),
                     Parcels_w_Units=('Units', lambda x: (x > 0).sum()),
                     Total_Units=('Units','sum'),
                 )
                 .reset_index())
    density['Pct_w_Units'] = (density['Parcels_w_Units'] / density['Parcels'] * 100).round(1)
    density['Avg_Units']   = (density['Total_Units'] / density['Parcels_w_Units']).round(2)
    print(density.to_string(index=False))

In [ ]:
if 'df_lost' in dir() and len(df_lost):
    df_lost_z = df_lost.copy()
    df_lost_z['Zone_ID']   = df_lost_z['APN'].map(
        lambda a: zone_lookup.get(a, (None, None))[0])
    df_lost_z['Zone_Desc'] = df_lost_z['APN'].map(
        lambda a: zone_lookup.get(a, (None, None))[1])
    df_lost_z['Zone_Label'] = df_lost_z['Zone_Desc'].fillna('No Zoning Match')

    by_zone = (df_lost_z.groupby(['Zone_Label','Issue_Category'])
                 .agg(APNs=('APN','count'), Units=('Total_Units_CSV','sum'))
                 .sort_values('Units', ascending=False))
    print("Lost APNs + Units by Zoning × Issue Category:")
    pd.set_option('display.max_colwidth', 60)
    print(by_zone.to_string())
else:
    print("df_lost not available — run cell 21 first")

### 14d. Lost APNs by Zoning

Cross-references the lost-APN list with the fresh zoning lookup to show which zoning
categories have the most outstanding unit deficits.

In [ ]:
# Null zoning: unique APNs (one row per APN, latest year)
df_latest = df_z.sort_values('Year', ascending=False).drop_duplicates('APN')
df_null_z = df_latest[df_latest['Zone_ID'].isna()].copy()

print(f"APNs with NO zoning match after fresh join: {len(df_null_z):,}")
print(f"  ({100*len(df_null_z)/len(df_latest):.1f}% of all {len(df_latest):,} unique APNs)\n")

# Break down by county
if 'County' in df_null_z.columns:
    by_county = (df_null_z.groupby('County')
                   .agg(APNs=('APN','count'), Units=('Units','sum'))
                   .sort_values('APNs', ascending=False))
    print("By County:")
    print(by_county.to_string())
    print()

# Break down by WITHIN_TRPA_BNDY
if 'WITHIN_TRPA_BNDY' in df_null_z.columns:
    by_trpa = (df_null_z.groupby('WITHIN_TRPA_BNDY')
                 .agg(APNs=('APN','count'), Units=('Units','sum')))
    by_trpa.index = by_trpa.index.map({0: 'Outside TRPA boundary', 1: 'Inside TRPA boundary'})
    print("By WITHIN_TRPA_BNDY:")
    print(by_trpa.to_string())

### 14c. Null Zoning Diagnostics — Where Are the Gaps?

Parcels that still have no zoning after the fresh INTERSECT join are either:
- Entirely outside all zoning polygons (outside TRPA boundary, Nevada state parcels, federal land)
- Geometry errors (slivers, invalid centroid placement)

The table below shows the null-zoning parcels broken down by county and WITHIN_TRPA_BNDY.

In [ ]:
df_z['ZID_Label'] = df_z['Zone_ID'].fillna('NONE')

piv_zid = (df_z.groupby(['ZID_Label', 'Year'])['Units']
             .sum()
             .unstack('Year')
             .fillna(0)
             .astype(int))
piv_zid['TOTAL'] = piv_zid.sum(axis=1)
piv_zid = piv_zid.sort_values('TOTAL', ascending=False)

print(piv_zid.head(40))

### 14b. Units by Zoning ID × Year

ZONING_ID is more granular than description — useful for spotting specific zone codes
that are gaining or losing units.

In [ ]:
df_z['Zone_Label'] = df_z['Zone_Desc'].fillna('No Zoning Match')

piv_zone = (df_z.groupby(['Zone_Label', 'Year'])['Units']
              .sum()
              .unstack('Year')
              .fillna(0)
              .astype(int))
piv_zone['TOTAL'] = piv_zone.sum(axis=1)
piv_zone = piv_zone.sort_values('TOTAL', ascending=False)

print(f"Zoning categories : {len(piv_zone)}")
print(f"  (including {(piv_zone.index == 'No Zoning Match').sum()} 'No Zoning Match' row)")
print()
print(piv_zone)

### 14a. Units by Zoning Description × Year

In [ ]:
# Apply fresh zoning to df — add Zone_ID and Zone_Desc columns
df_z = df.copy()
df_z['Zone_ID']   = df_z['APN'].map(lambda a: zone_lookup.get(a, (None, None))[0])
df_z['Zone_Desc'] = df_z['APN'].map(lambda a: zone_lookup.get(a, (None, None))[1])

# Diagnostic: compare FC-stored vs fresh-join coverage, split by WITHIN_TRPA_BNDY
if 'ZONING_ID' in df_z.columns and 'WITHIN_TRPA_BNDY' in df_z.columns:
    # One row per APN (use 2024 rows for comparison)
    df_ref = df_z[df_z['Year'] == JOIN_YEAR].drop_duplicates('APN')
    diag = []
    for flag, label in [(1, 'In TRPA boundary'), (0, 'Outside TRPA boundary'), (None, 'All')]:
        sub = df_ref if flag is None else df_ref[df_ref['WITHIN_TRPA_BNDY'] == flag]
        diag.append({
            'Subset'          : label,
            'APNs'            : len(sub),
            'FC_stored_pct'   : f"{100*sub['ZONING_ID'].notna().mean():.1f}%",
            'Fresh_join_pct'  : f"{100*sub['Zone_ID'].notna().mean():.1f}%",
        })
    pd.set_option('display.colheader_justify', 'left')
    print(pd.DataFrame(diag).to_string(index=False))
else:
    print("ZONING_ID or WITHIN_TRPA_BNDY not in df — skipping comparison")

In [ ]:
import time as _t

ZONING_SVC = "https://maps.trpa.org/server/rest/services/Zoning/MapServer/0"
JOIN_YEAR  = max(CSV_YEARS)   # use latest year for representative centroid per APN

_PT  = "memory/s14_centroids"
_SJ  = "memory/s14_zoning_join"
_LYR = "memory/s14_scope_lyr"
_ZLY = "memory/s14_zoning_lyr"

for _t_ in [_PT, _SJ, _LYR, _ZLY]:
    if arcpy.Exists(_t_): arcpy.management.Delete(_t_)

t0 = _t.time()
print(f"Building centroids from {OUTPUT_FC} (year={JOIN_YEAR}) ...")
arcpy.management.MakeFeatureLayer(OUTPUT_FC, _LYR, f"YEAR = {JOIN_YEAR}")
n_scope = int(arcpy.management.GetCount(_LYR).getOutput(0))
print(f"  {n_scope:,} parcels in scope")

arcpy.management.FeatureToPoint(_LYR, _PT, "INSIDE")

print("Spatial join → Zoning service (INTERSECT) ...")
arcpy.management.MakeFeatureLayer(ZONING_SVC, _ZLY)
arcpy.analysis.SpatialJoin(
    target_features=_PT,
    join_features=_ZLY,
    out_feature_class=_SJ,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT",
)

# Detect field names (may be suffixed _1 if they conflict with OUTPUT_FC fields)
_jf = {f.name for f in arcpy.ListFields(_SJ)}
_zid_f   = "ZONING_ID_1"   if "ZONING_ID_1"   in _jf else "ZONING_ID"
_zdsc_f  = "ZONING_DESCRIPTION_1" if "ZONING_DESCRIPTION_1" in _jf else "ZONING_DESCRIPTION"
print(f"  Reading fields: {_zid_f}, {_zdsc_f}")

# Build APN → (zone_id, zone_desc) lookup — one entry per APN
zone_lookup = {}
with arcpy.da.SearchCursor(_SJ, ["APN", _zid_f, _zdsc_f]) as _cur:
    for _apn, _zid, _zdsc in _cur:
        if _apn:
            zone_lookup[str(_apn).strip()] = (_zid, _zdsc)

for _t_ in [_PT, _SJ, _LYR, _ZLY]:
    if arcpy.Exists(_t_): arcpy.management.Delete(_t_)

n_matched = sum(1 for v in zone_lookup.values() if v[0] is not None)
print(f"\nResults ({_t.time()-t0:.0f}s):")
print(f"  APNs in scope    : {len(zone_lookup):,}")
print(f"  APNs with zoning : {n_matched:,}  ({100*n_matched/len(zone_lookup):.1f}%)")
print(f"  APNs null zoning : {len(zone_lookup)-n_matched:,}")

In [ ]:
import time as _t2

_LO_LYR  = "s14_lo_scope"
_ZO_LYR  = "s14_lo_zone"
_LO_JOIN = "memory/s14_lo_join"

for _x in [_LO_LYR, _ZO_LYR, _LO_JOIN]:
    if arcpy.Exists(_x): arcpy.management.Delete(_x)

t1 = _t2.time()
print(f"LARGEST_OVERLAP polygon join (year={JOIN_YEAR}) ...")
arcpy.management.MakeFeatureLayer(OUTPUT_FC, _LO_LYR, f"YEAR = {JOIN_YEAR}")
arcpy.management.MakeFeatureLayer(ZONING_SVC, _ZO_LYR)

arcpy.analysis.SpatialJoin(
    target_features=_LO_LYR,
    join_features=_ZO_LYR,
    out_feature_class=_LO_JOIN,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="LARGEST_OVERLAP",
)

_jf3    = {f.name for f in arcpy.ListFields(_LO_JOIN)}
_zid3   = "ZONING_ID_1"          if "ZONING_ID_1"          in _jf3 else "ZONING_ID"
_zdsc3  = "ZONING_DESCRIPTION_1" if "ZONING_DESCRIPTION_1" in _jf3 else "ZONING_DESCRIPTION"

zone_lookup_lo = {}
with arcpy.da.SearchCursor(_LO_JOIN, ["APN", _zid3, _zdsc3]) as _c:
    for _a, _z, _d in _c:
        if _a:
            zone_lookup_lo[str(_a).strip()] = (_z, _d)

for _x in [_LO_LYR, _ZO_LYR, _LO_JOIN]:
    if arcpy.Exists(_x): arcpy.management.Delete(_x)

n_lo = sum(1 for v in zone_lookup_lo.values() if v[0] is not None)

print(f"\n--- Coverage comparison (year {JOIN_YEAR}) ---")
print(f"  INTERSECT  centroid → polygon : {n_matched:>6,} / {len(zone_lookup):,}  "
      f"({100*n_matched/len(zone_lookup):.1f}%)")
print(f"  LARGEST_OVERLAP polygon → polygon : {n_lo:>6,} / {len(zone_lookup_lo):,}  "
      f"({100*n_lo/len(zone_lookup_lo):.1f}%)")
print(f"  Improvement : +{n_lo - n_matched:,} APNs  ({_t2.time()-t1:.0f}s)")

### 14e. Coverage Test — INTERSECT (centroid) vs LARGEST_OVERLAP (polygon)

Tests whether switching from centroid-point INTERSECT to polygon-polygon LARGEST_OVERLAP
improves zoning coverage.  LARGEST_OVERLAP assigns each parcel the zone it overlaps most,
so gaps between adjacent zone polygons can never swallow a result.

---
## 14. Zoning Analysis — Fresh Spatial Join

Re-runs the centroid → Zoning service join directly in the notebook so results can be
inspected interactively.  Uses **INTERSECT** (more permissive than the ETL's
`HAVE_THEIR_CENTER_IN`) to maximise coverage, then compares to the values already stored
in the FC to diagnose the 71 % null rate.